In [3]:
# OpenAQ PM2.5 sensor configuration
SENSOR_ID = 1097
LOCATION_ID = 625

print("OpenAQ PM2.5 configuration loaded")
print("Station: Manhattan / IS143")
print("Location ID:", LOCATION_ID)
print("Sensor ID:", SENSOR_ID)

StatementMeta(, bbff176a-ead3-4b05-9ecf-cb750524f682, 5, Finished, Available, Finished, False)

OpenAQ PM2.5 configuration loaded
Station: Manhattan / IS143
Location ID: 625
Sensor ID: 1097


In [4]:
import requests
import pandas as pd

url = "https://api.openaq.org/v3/sensors/1097/days"

headers = {
    "X-API-Key": "ebd054f062ded73867e482d13caef0221b10c4b72d8c4165412fc9a6022577bb"
}

params = {
    "date_from": "2026-01-01",
    "date_to": "2026-01-31",
    "limit": 1000,
    "page": 1
}

response = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=60
)

print("Status:", response.status_code)
response.raise_for_status()

records = response.json()["results"]

rows = []

for item in records:
    rows.append({
        "date": item["period"]["datetimeFrom"]["local"][:10],
        "station": "Manhattan/IS143",
        "location_id": 625,
        "sensor_id": 1097,
        "parameter": "pm25",
        "value": item["value"],
        "unit": "µg/m³",
        "coverage_percent": item["coverage"]["percentComplete"]
    })

air_df = pd.DataFrame(rows)

print("Total records:", len(air_df))
print("Unique dates:", air_df["date"].nunique())

display(air_df)

StatementMeta(, bbff176a-ead3-4b05-9ecf-cb750524f682, 6, Finished, Available, Finished, False)

Status: 200
Total records: 31
Unique dates: 31


SynapseWidget(Synapse.DataFrame, 35f98dfd-15cd-4212-a7e4-9cee3f3c97a8)

In [5]:
# Convert Pandas DataFrame to Spark DataFrame
spark_df = spark.createDataFrame(air_df)

# Save as Delta table in Bronze Lakehouse
(
    spark_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("openaq_pm25_daily")
)

print("OpenAQ data saved successfully!")
print("Total records:", spark_df.count())

StatementMeta(, bbff176a-ead3-4b05-9ecf-cb750524f682, 7, Finished, Available, Finished, False)

OpenAQ data saved successfully!
Total records: 31


In [6]:
spark.sql("SHOW TABLES").show(truncate=False)

StatementMeta(, bbff176a-ead3-4b05-9ecf-cb750524f682, 8, Finished, Available, Finished, False)

+----------------------------------------+--------------------+-----------+
|namespace                               |tableName           |isTemporary|
+----------------------------------------+--------------------+-----------+
|NYC_Mobility_Analytics.NYC_Bronze_LH.dbo|ecb_usd_eur_raw     |false      |
|NYC_Mobility_Analytics.NYC_Bronze_LH.dbo|openaq_pm25_daily   |false      |
|NYC_Mobility_Analytics.NYC_Bronze_LH.dbo|taxi_zone_lookup_raw|false      |
|NYC_Mobility_Analytics.NYC_Bronze_LH.dbo|worldbank_gdp_raw   |false      |
+----------------------------------------+--------------------+-----------+

